# kwargs-pass-through-recipe — worked example 3: Recipe kwargs are call-site faithful — no injected defaults

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kwargs-pass-through-recipe`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The Recipe must store the kwargs exactly as they were passed at the call site, not the function's internal defaults. If a caller writes `wrapped_relu(x)` with no kwargs, the Recipe should store `{}`. Injecting the function's default values (e.g., `keepdims=False`) would break backward functions that key on the presence vs absence of a kwarg.

## Worked solution

Step 1: Implement `wrap_forward_fn` that stores `dict(kwargs)` — a copy of the call-site kwargs.

Step 2: Call the wrapper three times on the same input: once with no kwargs, once with `axis=1`, and once with `axis=1, keepdims=True`.

Step 3: Print each Recipe's kwargs and verify they match the call site exactly: `{}`, `{'axis': 1}`, `{'axis': 1, 'keepdims': True}`.

Step 4: Confirm that no extra keys appear in the first call's Recipe — in particular, that numpy's own defaults are not injected.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        # Store a copy of kwargs — call-site faithful, no defaults injected
        out.recipe = Recipe(fwd_fn, raw_args, dict(kwargs), parents)
        return out
    return tensor_func

wrapped_sum = wrap_forward_fn(np.sum)
x = MiniTensor(np.arange(8.0).reshape(2, 4))

# Three calls, three different kwargs
out0 = wrapped_sum(x)
out1 = wrapped_sum(x, axis=1)
out2 = wrapped_sum(x, axis=1, keepdims=True)

print('call 0 kwargs:', out0.recipe.kwargs)   # {}
print('call 1 kwargs:', out1.recipe.kwargs)   # {'axis': 1}
print('call 2 kwargs:', out2.recipe.kwargs)   # {'axis': 1, 'keepdims': True}

assert out0.recipe.kwargs == {}
assert out1.recipe.kwargs == {'axis': 1}
assert out2.recipe.kwargs == {'axis': 1, 'keepdims': True}
print('All kwargs stored faithfully.')